# 2107541 Machine Learning for Environmental Applications
## Individual Project : Predicting Water Turbidity with Total Suspended Solids (TSS) 
### Done by : 6730084521 Chatrphol Ovanonchai

#### Disclaimer : Mostly 95% of this colab were done by myself , but use some instructions/guide from AI sometimes

## Oveview 
These are s

## Part 0 : Setting Environments
In this process , I'll import libraries & required for this project

In [37]:
import numpy as np , pandas as pd , seaborn as sns , matplotlib.pyplot as plt
import os , warnings

## import model 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import GroupKFold , GroupShuffleSplit , cross_val_predict 
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb , shap 
from xgboost import XGBRegressor

## ignore warnings 
warnings.filterwarnings("ignore")

### Caution : Dataset is too large to upload to GitHub , so I need to upload via Google Drive Instead
> You can access via this link [Dataset AquaSat Landset Water Quality](https://drive.google.com/file/d/1zQtaYk-Im-lwZ0icy0dVEETFqwlJVjiW/view?usp=sharing)

In [38]:
## read dataset
df = pd.read_csv("Resources/Dataset/aquasat_landsat_lake_water_quality.csv")

In [39]:
df.head()

,system:index,SiteID,blue,blue_sd,date_unity,green,green_sd,nir,nir_sd,path,...,long,TZID,date_utc,clouds,time,landsat_id,timediff,pwater,type,id
0,1,8348,230.0,25.890351,2004-08-23T12:00:00Z,243.0,22.781871,383.000000,30.704239,10,...,-67.829399,America/New_York,2004-08-23T16:00:00Z,17.0,2004-08-23T15:02:20Z,LE07_L1TP_010028_20040823_20160926_01_T1,0.961041,97.0,Lake,New
1,3,8348,200.0,23.917018,2001-08-22T12:00:00Z,212.0,23.645332,242.000000,23.730330,10,...,-67.829399,America/New_York,2001-08-22T16:00:00Z,6.0,2001-08-23T14:53:48Z,LT05_L1TP_010028_20010823_20160917_01_T1,-22.896826,97.0,Lake,New
2,4,8348,543.0,31.119580,2003-06-25T12:00:00Z,495.0,31.446904,639.000000,33.756642,10,...,-67.829399,America/New_York,2003-06-25T16:00:00Z,3.0,2003-06-26T14:49:37Z,LT05_L1TP_010028_20030626_20160915_01_T1,-22.827122,97.0,Lake,New
3,6,7889,269.0,31.265990,1988-07-18T12:00:00Z,270.0,32.082510,364.470719,56.942639,10,...,-67.585147,America/Moncton,1988-07-18T15:00:00Z,26.0,1988-07-18T14:44:16Z,LT05_L1TP_010028_19880718_20161002_01_T1,0.262185,95.0,Lake,New
4,7,7889,146.0,18.720043,1990-07-08T12:00:00Z,137.0,33.139252,212.000000,17.413788,10,...,-67.585147,America/Moncton,1990-07-08T15:00:00Z,4.0,1990-07-08T14:33:41Z,LT05_L1TP_010028_19900708_20160930_01_T1,0.438419,95.0,Lake,New


## Part I : Data Preprocessing & Feature Engineering

### Step 0 : Data Properties

In [40]:
## see shape and properties

print("Dataset shape : " , df.shape)
df.describe()

## check null value (ony on some columns)
print(df[['tss','clouds','pwater','qa','chl_a','doc','secchi','tis','p_sand']].isnull().sum()) 
    # missing value on TSS = 443613 of 603432 

Dataset shape :  (603432, 43)
tss       443613
clouds         0
pwater         0
qa             0
chl_a     383240
doc       561014
secchi    113989
tis       590768
p_sand    602749
dtype: int64


### Step 1 : Data Cleaning
Clean out conditions below
- Unused Columns : `["chl_a" , "doc" , "secchi" , "tis" , "p_sand"]`

- Invalidate Surface Reflection : Reflection < 0 `OR` Reflection > 10000

- Unclear water from satellite : `qa` not in code 68 or 324

- Water is too close to coast : `pwater should >= 50`

In [41]:
## It's time to clean some unused & misorganized value -> store in df_cleaned

BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
clean_log = []

## create function 
def keep_df(data , cond, step):
    out = data[cond]
    
    ## add data to clean_log to revert back when error 
    clean_log.append((step, len(data), len(out), len(data) - len(out)))
    print(f"{step:<20} {len(data):>7,} -> {len(out):>7,}  (removed {len(data) - len(out):,})")
    return out

## step 1 : drop unused features 
df_cleaned = df.drop(columns = ["chl_a" , "doc" , "secchi" , "tis" , "p_sand"])

## step 2 : clean rows havong NA values on TSS out 
condition_tss = (df_cleaned["tss"].notna()) & (df_cleaned['tss'] > 0)
df_cleaned = keep_df(df_cleaned , condition_tss , "No TSS value condition : ")

## step 3 : clean row having error surface reflection
condition_valid_reflection = ((df_cleaned[BANDS] > 0) & (df_cleaned[BANDS] <= 10000)).all(axis = 1)
    ## reflection_percent should >0 and <100 percent
df_cleaned = keep_df(df_cleaned , condition_valid_reflection , "Valid reflection condition : ")

## step 4 : clean row having unclear water out (is in code 68 or 324 from satellite)

    ## pixel QA : 68 = clear water (Landsat 5/7), 324 = clear water (Landsat 8)
condition_clear_water = df_cleaned["qa"].round().isin([68, 324])
df_cleaned = keep_df(df_cleaned , condition_clear_water , "Clear Water from Satellite Condition : ")

## step 5 : clean row that is near coast (pwater >= 50)
condition_near_coast = (df_cleaned['pwater'] >= 50)
df_cleaned = keep_df(df_cleaned , condition_near_coast , "Near Coast Condition : ")

No TSS value condition :  603,432 -> 159,819  (removed 443,613)
Valid reflection condition :  159,819 -> 157,420  (removed 2,399)
Clear Water from Satellite Condition :  157,420 -> 135,893  (removed 21,527)
Near Coast Condition :  135,893 -> 128,603  (removed 7,290)


### Step 2 : Feature Engineering

Create new column with existing column to calculate more easier

- Surface Reflection (Bands) : multiply `1e-4` to get real value

- TSS : convert to logarithmic scale (to prevent right skewed)

- Create satellite normalized index & band ratio : `NDTI` , `NDWI` , `NDSSI` , `R/G` , `G/B`

- Brightness : create `log_visible_bands` from log(r+g+b) because turbid water will be brighter every band 

- Datetime : create `doy_sin` and `doy_cos` (doy is from date of year)
> to discriminate seasons that can affect to TSS value



In [42]:
## transform TSS to log(1 + TSS) to prevent right-skewed graph 
df_cleaned['tss_log'] = np.log1p(df_cleaned['tss'])

## satellite indexes
BANDS_SD = ["blue_sd", "green_sd", "red_sd", "nir_sd", "swir1_sd", "swir2_sd"]

df_cleaned[BANDS_SD] = df_cleaned[BANDS_SD] * 1e-4
df_cleaned[BANDS] = df_cleaned[BANDS] * 1e-4

## define pd.Series as variables for easier notation
b , g , r , n = (df_cleaned[col] for col in ['blue','green','red','nir'])

df_cleaned["NDTI"] = (r - g) / (r + g)
df_cleaned["NDWI"] = (g - n) / (g + n)
df_cleaned["NDSSI"] = (b - n) / (b + n)

df_cleaned["R_G"], df_cleaned["R_B"] = r / g, r / b
df_cleaned["G_B"], df_cleaned["NIR_R"] = g / b, n / r
df_cleaned["log_visible"] = np.log(b + g + r)

doy = pd.to_datetime(df_cleaned["date"]).dt.dayofyear
df_cleaned["doy_sin"] = np.sin(2 * np.pi * doy / 365)
df_cleaned["doy_cos"] = np.cos(2 * np.pi * doy / 365)

## Part II : Setting Model Properties
We have to set model properties first before start training , these are steps I would do

Note : before `train_test_split` process , we shouldn't use `train_test_split` for each row , should divide with `SiteID` instead
> Reason : Causing `Data Leakage` from same `SiteID` , using `GroupShuffleSplit` and `GroupKFold` is better solution

Steps to set model properties 
- Define Feature (X) and Targets (Y) 

- Split Train/Test by SiteID using `GroupShuffleSplit` (80/20)

In [ ]:
## step 1 : define features & targets

features = df_cleaned[['blue', 'green', 'red', "nir" , "swir1", "swir2"] + ["NDTI", "NDWI", "NDSSI", "R_G", "R_B", "G_B", "NIR_R", "log_visible", "doy_sin", "doy_cos"]]
targets = df_cleaned["tss_log"] # Series

## define groups to do GroupShuffleSplit (similar as train_test_split)
groups = df_cleaned["SiteID"] # should be Series , not DataFrame

## train data & test data splitting (n_splits = 1 for test data --> from the reason that we need siteID to be different in all time)
gss = GroupShuffleSplit(n_splits = 1 , train_size = 0.8 , test_size = 0.2 , random_state = 42) # 42 for lock state

## get train & test idx
train_idx, test_idx = next(gss.split(features, targets, groups))

X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
y_train, y_test = targets.iloc[train_idx], targets.iloc[test_idx]

g_train = groups.iloc[train_idx]

## Part III : Model Training

- Setting Pipeline for model & set 3+1 models `LinearRegression` (Baseline) , `RandomForestRegressor` , `XGBRegressor` to train 
> Note : Use `Pipeline` with `StandardScaler` for Linear Regression (tree models don't need scaling)
> Note 2 : Adding new `Neural Network` Model to train deeper (with Multi-Layer Perceptron (MLP))

- Setting Hyperparameters (will do hyperparameter tuning in Part IV)

### Step I : Pipeline for each model

In [ ]:
### Step 1 : Pipeline for each model
baselne_lin_reg_pipeline = Pipeline([
    ("scaler" , StandardScaler()) , 
    ("model"  , LinearRegression()) 
] , memory = None)

random_forest_pipeline = Pipeline([ 
    ("model"  , RandomForestRegressor(n_estimators = 300 , min_samples_leaf = 3 , n_jobs = -1 , random_state = 42 , max_features = 16) ) # njobs = -1 => get all CPU cores to compute
] , memory = None)

xgboost_pipeline = Pipeline([
    ("model" , XGBRegressor(n_estimators = 300 , max_depth = 4 , learning_rate = 0.05 , subsample = 0.8 , colsample_by_tree = 0.8) )
] , memory = None)

## extra model : neural network 
neural_mlp_pipeline = Pipeline(
    ("scaler" , StandardScaler()) , # use scaler from gradient_descent  
    ("model" , XGBRegressor(n_estimators = 300 , max_depth = 4 , learning_rate = 0.05 , subsample = 0.8 , colsample_by_tree = 0.8) )
] , memory = None)

### Step II : 

## Part IV : Optimize Data & Model 